## Lab 04 / Step 4 : Reinforcement Learning (RL) of Large Language Models (LLMs)

## Task : from a prompt, generate a response

## Use pre-trained models:
### SFT-LLM model from Step 2 as reference model to fine-tune w.r.t. rank reward
### SL-RM model from Step 3 as reward model to evaluate generation

## Data structure : prompt

### Xavier Bresson

### Number of data points for GPT-3, 175B parameters
+ Step #1 : 300B tokens
+ Step #2 : 10K-100k pairs (prompt, response)
+ Step #3 : 100K-1M triples (prompt, positive response, negative response)
+ Step #4 : 10K-100K prompts

### Number of data points for this tutorial
+ Step #1 : 200M tokens
+ Step #2 : 200K pairs (prompt, response)
+ Step #3 : 100K triples (prompt, positive response, negative response)
+ Step #4 : 100 prompts

### Objectives
+ Adapt PPO reinforcement learning technique to language generation
+ Train with batch of prompt training for fast training with GPU
+ Use pre-trained models from steps 2 and 3


In [1]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/IPAM_Mar26_codes/labs_lecture09'
    print(path_to_file)
    # move to Google Drive directory
    os.chdir(path_to_file)
    !pwd
    

In [2]:
# Libraries
import torch
print(torch.__version__)
import torch.nn as nn
import torch.optim as optim
import time
import matplotlib.pyplot as plt
import logging
logging.getLogger().setLevel(logging.CRITICAL) # remove warnings
import os, datetime


2.1.2


## Time stamp for save/load data


In [3]:
# save time stamp
time_stamp = datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S")

# check dataset folder exists
data_dir = os.path.join('dataset')
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# select a time stamp
use_saved_time_stamp = False
use_saved_time_stamp = True
if use_saved_time_stamp:
    time_stamp = '26-02-07--18-48-32' # trained on GPU on xxx

print('time_stamp:', time_stamp, '\n')


time_stamp: 26-02-07--18-48-32 



## Load dictionary of tokens from Step 1


In [4]:
# load_file_dictionary = 'dataset/step1_SSL_dictionary_' + str(time_stamp) + '.pt'
file_dictionary = 'dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt' # GPU pre-trained
print('file_dictionary:', file_dictionary, '\n')
import os
os.makedirs('dataset', exist_ok=True)
if not os.path.exists(file_dictionary):
    print(f'Downloading {file_dictionary} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/jvb5xkk0kfzwrm7vzha3o/step1_SSL_dictionary_26-02-07-18-48-32_200M.pt?rlkey=tprax7nqqpvogo04fhe7m2okd&dl=1"
    !wget -q "{dropbox_url}" -O "{file_dictionary}"
dictionary, num_tokens, token2index, index2token = torch.load(file_dictionary) # load dictionary of tokens
print('dictionary:',dictionary,'\n')
print('num_tokens (unique):',num_tokens,'\n')
print('token2index:', token2index,'\n')
print('index2token:', index2token,'\n')
func_tokens2indices = lambda list_tokens: [token2index[token] for token in list_tokens] # ['Let', '5', 'be', 'the'] => [113, 46, 114, 115]
func_indices2tokens = lambda list_ints: [index2token[integer] for integer in list_ints] # [113, 46, 114, 115] => ['Let', '5', 'be', 'the']
func_str2tokens = lambda input_str: [token_str for token_str in input_str.split()]      # 'Let 5 be the' => ['Let', '5', 'be', 'the']
func_tokens2str = lambda list_str: ' '.join(list_str)                                   # ['Let', '5', 'be', 'the'] => 'Let 5 be the'


file_dictionary: dataset/step1_SSL_dictionary_26-02-07--18-48-32_200M.pt 

dictionary: ['51', '57', '63', '69', '75', '81', '87', '93', '99', '<SEP>', '91', '1', '2', '3', '4', '5', '6', '7', '8', '9', '77', '79', '83', '85', '89', '48', '55', '62', '76', '90', '97', '29', '38', '47', '56', '65', '74', '92', '94', '100', '50', '64', '71', '78', '39', '95', '46', '53', '60', '67', '88', '13', '22', '31', '40', '49', '58', '41', '42', '43', '44', '30', '54', '15', '36', '72', '86', '96', '18', '19', '20', '21', '23', '24', '25', '26', '27', '28', '73', '98', '82', '52', '34', '66', '70', '68', '84', '37', '61', '11', '14', '17', '32', '59', '10', '12', '16', '80', '35', '33', '45', '0', 'generate', 'an', 'arithmetic', 'series', 'with', 'terms', 'starting', 'value', 'and', 'common', 'difference', 'Let', 'be', 'the', 'number', 'of', 'then', 'write', 'make', 'a', 'type', 'which', 'starts', 'at', 'elements', '<PAD>', '<EOS>'] 

num_tokens (unique): 129 

token2index: {'51': 0, '57': 1, '63':

## Generate the RL training set of prompts


In [5]:
# generate arithmetic series
max_value = 100 # maximum value in the sequence
def arithmetic_series(max_value, s, d, n):
    seq = []
    for i in range(n):
        v = s + i * d
        if v <= max_value:
            seq.append(v)
        else:
            break
    return seq

# generate training data, i.e. list of prompts
#   prompt = [ generate arithmetic series of 5 terms with difference 2 starting at 3 ]
save_training_data = False
save_training_data = True
if save_training_data:

    # "collect" training set
    list_prompt_RL = []
    num_training_data = 100 # number of prompts
    start = time.time()
    for idx in range(num_training_data):

        # parameters for arithmetic series
        s = torch.randint(low=0, high=max_value, size=(1,)).item() # starting integer of the series
        d = torch.randint(low=1, high=10, size=(1,)).item() # value of common difference
        n = torch.randint(low=5, high=15, size=(1,)).item() # number of element in the series
        #print('max_value: %d, start_value: %d, common_difference: %d, number_of_terms: %d' % (m,s,d,n))

        # generate prompt : sample a prompt between 3 candidate prompts
        prompt = {}
        prompt[1] = 'generate an arithmetic series with ' + str(n) + ' terms starting with value ' + str(s) + ' and common difference ' + str(d)
        prompt[2] = 'make a series of arithmetic type which starts at ' + str(s) + ' with ' + str(n) + ' elements and ' + str(d) + ' common difference value'
        prompt[3] = 'Let ' + str(n) + ' be the number of terms ' + str(s) + ' the starting number and ' + str(d) + ' the common difference then write the arithmetic series'
        random_int = torch.randint(low=1, high=3+1, size=(1,)).item() # random number in {1,2,3}
        #random_int = 1 # debug
        prompt = prompt[random_int]
        # response = arithmetic_series(m,s,d,n)

        # covert from token to pytorch
        prompt = [str(i) for i in prompt.split()] # convert a string into seq of tokens (w/ string type)
        prompt = func_tokens2indices(prompt) # convert from token (str) to index (int)
        prompt = torch.tensor(prompt) # convert to pytorch

        # append
        list_prompt_RL.append(prompt)

        # track
        if not idx%1000:
            print('idx: %d, time(sec): %.3f' % (idx, time.time()-start) )

    # print
    print('number of training data / prompt :',len(list_prompt_RL),'\n')
    for idx, prompt in enumerate(list_prompt_RL[:4]):
        prompt = func_tokens2str(func_indices2tokens(prompt.tolist()))
        print('training_set[%d] : %s ' % (idx, prompt) , '\n' )

    # save training data
    save_file = data_dir + '/step4_RL_training_set_' + time_stamp + '.pt'
    print('save_file:', save_file, '\n')
    torch.save([list_prompt_RL],save_file) # save list of prompts

else:

    # load training data
    load_file = data_dir + '/step4_RL_training_set_' + time_stamp + '.pt'
    print('load_file:', load_file, '\n')
    list_prompt_RL = torch.load(load_file)[0]

    # print
    print('number of training data / prompt :',len(list_prompt_RL),'\n')
    for idx, prompt in enumerate(list_prompt_RL[:3]):
        prompt = func_tokens2str(func_indices2tokens(prompt.tolist()))
        print('training_set[%d] : %s ' % (idx, prompt) , '\n' )


# batching parameters
num_prompt_RL = len(list_prompt_RL) # number of prompt sequences
print('num_prompt_RL: %d\n' % (num_prompt_RL))


idx: 0, time(sec): 0.002
number of training data / prompt : 100 

training_set[0] : Let 11 be the number of terms 16 the starting number and 1 the common difference then write the arithmetic series  

training_set[1] : Let 11 be the number of terms 79 the starting number and 9 the common difference then write the arithmetic series  

training_set[2] : generate an arithmetic series with 12 terms starting with value 93 and common difference 5  

training_set[3] : Let 7 be the number of terms 52 the starting number and 4 the common difference then write the arithmetic series  

save_file: dataset/step4_RL_training_set_26-02-07--18-48-32.pt 

num_prompt_RL: 100



## Ground truth reward

In [6]:
import re
def extract_arithmetic_params(prompt):
    # Pattern 1: 'generate an arithmetic series with n terms starting with value s and common difference d'
    pattern1 = r"generate an arithmetic series with (\d+) terms starting with value (\d+\.?\d*) and common difference (\d+\.?\d*)"
    # Pattern 2: 'make a series of arithmetic type which starts at s with n elements and d common difference value'
    pattern2 = r"make a series of arithmetic type which starts at (\d+\.?\d*) with (\d+) elements and (\d+\.?\d*) common difference value"
    # Pattern 3: 'Let n be the number of terms s the starting number and d the common difference then write the arithmetic series'
    pattern3 = r"Let (\d+) be the number of terms (\d+\.?\d*) the starting number and (\d+\.?\d*) the common difference then write the arithmetic series"
    match1 = re.search(pattern1, prompt)
    if match1:
        n = int(match1.group(1))
        s = int(match1.group(2)) 
        d = int(match1.group(3))
        return n, s, d
    match2 = re.search(pattern2, prompt)
    if match2:
        s = int(match2.group(1))
        n = int(match2.group(2))
        d = int(match2.group(3))
        return n, s, d
    match3 = re.search(pattern3, prompt)
    if match3:
        n = int(match3.group(1))
        s = int(match3.group(2))
        d = int(match3.group(3))
        return n, s, d
    return None # if no match

def reward_model_oracle(prompt, generated_response):
    prompt = func_tokens2str(func_indices2tokens(prompt.tolist()))
    generated_response = func_tokens2str(func_indices2tokens(generated_response))
    generated_response = [int(x) if x.isdigit() else 50 for x in generated_response.split()]
    n, s, d = extract_arithmetic_params(prompt)
    s = min(s, max_value); d = min(d, 10); n = max(5, min(n, 15))
    exact_response = arithmetic_series(max_value, s, d, n)
    exact_response_out = exact_response.copy()
    exact_response = torch.tensor(exact_response)
    generated_response = torch.tensor(generated_response)
    max_size = max(exact_response.size(0), generated_response.size(0)) # e.g. 4
    if len(generated_response)>len(exact_response): # penalize more the reward = 1
        exact_response = nn.functional.pad(exact_response,(0, max_size-exact_response.size(0)), 'constant', 1e3) # padding to get same-size vectors
        # no change to the generated_response
    else: 
        # no change to the exact_response
        generated_response = nn.functional.pad(generated_response,(0, max_size-generated_response.size(0)), 'constant', 100)
    r_min = 1; r_max = 7
    reward = r_min + (r_max - r_min) * ( 1 + 0.1*( (exact_response - generated_response).abs() ).float().sum().sqrt() )**(-1) 
    reward = reward.item()
    return reward, exact_response_out

idx = 25
# idx = torch.randint(0, len(list_prompt_RL), (1,)).item()
print('idx :', idx)
prompt = list_prompt_RL[idx]
# print('prompt:', prompt)
prompt_print = func_tokens2str(func_indices2tokens(prompt.tolist())) # -> exact_response = [22, 24, 26, 28, 30]
print('prompt_print :', prompt_print)
generated_response = [29, 35, 41, 47, 53, 59, 65, 71, 77] # exact_response for idx = 25
# generated_response = [29, 35, 41, 47, 53, 59, 65] 
print('generated_response :', generated_response)
generated_response = func_tokens2indices(func_str2tokens(" ".join([str(n) for n in generated_response])))
reward, exact_response = reward_model_oracle(prompt, generated_response)
print('exact_response :', exact_response)
print('reward :', reward)


idx : 25
prompt_print : Let 6 be the number of terms 61 the starting number and 4 the common difference then write the arithmetic series
generated_response : [29, 35, 41, 47, 53, 59, 65, 71, 77]
exact_response : [61, 65, 69, 73, 77, 81]
reward : 1.9330577850341797


## Transformers backbone from Step 2, Step 3 and Step 4


In [7]:
torch.manual_seed(0) # use same initial seed for reproducibility

# compute number of network parameters
def number_param(net):
    nb_param = 0
    for param in net.parameters():
        nb_param += param.numel()
    return nb_param

# GPU training
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    device = torch.device("cuda") # use GPU
else:
    device = torch.device("cpu")
print('device:',device,'\n')

# token embedding layer : convert seq of integers to seq of vectors
class token2vec(nn.Module):
    def __init__(self, num_tokens, d):
        super().__init__()
        self.token2vec = nn.Embedding(num_tokens, d) # map integer to one-hot vector (num_tokens dimensions), and project vector to d-dimentional space
    def forward(self, batch_int):
        batch_vec = self.token2vec(batch_int) # size=[batch_size, batch_length, d]
        return batch_vec

# multiple attention heads layer
class multiple_head_attention(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        d_head = d // num_heads
        assert d == d_head * num_heads # check divisiblity
        self.MHA = nn.MultiheadAttention(d, num_heads, batch_first=True)
        self.mask = torch.tril(torch.ones(context_length, context_length))==0 # mask to make attention to previous tokens only : { token(<=t) }, size=(context_length,context_length)
                   # torch.tril(ones) = True in the up-right part, True means *no* attention allowed in pytorch implementation
        self.context_length = context_length
    def forward(self, H):
        if H.size(1) == self.context_length: # training
            attn_mask = self.mask
        else: # when batch_length not= context_length, e.g. inference time / sequence generation
            current_batch_length = H.size(1)
            attn_mask = torch.tril(torch.ones(current_batch_length, current_batch_length))==0
        H_heads = self.MHA(H, H, H, attn_mask=attn_mask.to(device))[0] # pytorch implementation, size=[batch_size, batch_length, d]
        return H_heads

# Transformer block layer
class TransformerBlock(nn.Module):
    def __init__(self, d, context_length, num_heads):
        super().__init__()
        self.MHA = multiple_head_attention(d, context_length, num_heads)
        self.LN_MHA = nn.LayerNorm(d)
        self.MLP = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,d))
        self.LN_MLP = nn.LayerNorm(d)
    def forward(self, H):
        H = H + self.MHA(self.LN_MHA(H)) # size=[batch_size, batch_length, d]
        H = H + self.MLP(self.LN_MLP(H)) # size=[batch_size, batch_length, d]
        return H


device: cpu 



## Load pre-trained SFT-LLM network from Step 2


In [8]:
####################################
# Load pre-trained SFT-LLM net (Step 2)
####################################
class SFT_LLM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.seq_pos_encoding = torch.arange(context_length, device=device) # positional encoding = {0,1,2,...,context_length-1}
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.transformer_blocks = nn.ModuleList([ TransformerBlock(d, context_length, num_heads) for _ in range(num_layers) ]) # multiple transformer block layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
        self.context_length = context_length
        self.padding = padding_int
        self.eos = eos_int
    # Note : No forward function is needed
    
# Initialize RL-LLM with pre-trained SFT-LLM network (Step 2)
# checkpoint_file = 'checkpoint/step2_checkpoint_SFT_LLM_' + str(time_stamp) + '.pkl'
checkpoint_file = 'checkpoint/step2_checkpoint_SFT_LLM_26-02-07--18-48-32_200K_10M.pkl'
print(f'SFT-LLM net (Step 2) -- checkpoint_file: {checkpoint_file}')
checkpoint = torch.load(checkpoint_file, map_location=device)
net_parameters = checkpoint['net_parameters']
num_tokens = net_parameters['num_tokens']
d = net_parameters['d']
num_heads = net_parameters['num_heads']
context_length = net_parameters['context_length']
num_layers = net_parameters['num_layers']
padding_int = net_parameters['padding_int']
eos_int = net_parameters['eos_int']
padding_int = torch.tensor([func_tokens2indices('<PAD>'.split())[0]]).to(device) # end-of-sentence token for batch
eos_int = torch.tensor([func_tokens2indices('<EOS>'.split())[0]]).to(device) # end-of-sentence token for batch
SFT_LLMnet = SFT_LLM(num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int).to(device)
SFT_LLMnet.load_state_dict(checkpoint['SFT_LLMnet_dict']) # load pre-trained SFT-LM network (step 2)
num_param = number_param(SFT_LLMnet)
print('num_tokens: %d, d: %d, context_length: %d, num_heads: %d, num_layers: %d, padding_int: %d, eos_int: %d, num_param: %f' \
      % (num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int, num_param/1e6) )

checkpoint_SFT_LLM = checkpoint


SFT-LLM net (Step 2) -- checkpoint_file: checkpoint/step2_checkpoint_SFT_LLM_26-02-07--18-48-32_200K_10M.pkl
num_tokens: 129, d: 384, context_length: 40, num_heads: 6, num_layers: 6, padding_int: 127, eos_int: 128, num_param: 10.761345


## Load pre-trained RM network (reward model) from Step 3


In [9]:
class SL_RM(nn.Module):
    def __init__(self, SFT_LLM, d, context_length, padding_int, eos_int):
        super().__init__()
        self.SFT_LLM = SFT_LLM # SFT-LLM backbone 
        self.reward_prediction = nn.Sequential(nn.Linear(d,4*d), nn.SiLU(), nn.Linear(4*d,1)) # reward prediction layer
        self.cls_token_input = torch.tensor([0]).long().to(device)
        self.cls_token_embed = nn.Sequential( nn.Embedding(1, 4*d), nn.SiLU(), nn.Linear(4*d, d) )
        self.context_length = context_length
        self.padding = padding_int
        self.eos = eos_int
    def forward(self, final_sequence): 
        sequence = [ self.padding.item() ] * max(len(final_sequence),self.context_length)
        # print('sequence', sequence)
        sequence[-len(final_sequence):] = final_sequence
        # print('sequence', sequence)
        offset_start = max( self.context_length - len(final_sequence), 0 )
        # print('offset_start', offset_start)
        sequence_torch = torch.tensor(sequence).to(device)
        # print('sequence_torch', sequence_torch)
        context = sequence_torch[-self.context_length:] # crop if necessary the generated sequence with the context_length of the network
        # print('context', context)
        context = context.unsqueeze(0).to(device) # [bs=1, len(state), d]
        H = self.SFT_LLM.token2vec(context) + self.SFT_LLM.PE_embedding(self.SFT_LLM.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # [bs=1, len(context), d] [1, 15, 384]
        h_cls = self.cls_token_embed(self.cls_token_input).expand(1,-1,-1) # [1, 1, d]  
        H = torch.cat([H, h_cls], dim=1) # [bs=1, len(context)+1, d]
        for transformer_block in self.SFT_LLM.transformer_blocks: H = transformer_block(H) # [bs=1, len(context)+1, d]
        token_scores = H[:,-1,:] # extract last token scores to predict value [bs=1, d]
        reward_scores = self.reward_prediction(token_scores).squeeze() # compute reward scores, size=[]
        return reward_scores
        
# checkpoint_file = 'checkpoint/step3_checkpoint_SL_RM_' + time_stamp + '.pkl'
checkpoint_file = 'checkpoint/step3_checkpoint_SL_RM_26-02-07--18-48-32_100K_10M.pkl'
checkpoint = torch.load(checkpoint_file, map_location=device)
reward_model = SL_RM(SFT_LLMnet, d, context_length, padding_int, eos_int).to(device)
reward_model.load_state_dict(checkpoint['SL_RMnet_dict']) # load pre-trained SL-RM network from step #3
num_param = number_param(reward_model)
print(f'Reward model (Step 3), num_param: {num_param/1e6}\n')


Reward model (Step 3), num_param: 11.945986



## Step 4 : Define Policy (Actor) Net Value (Critic) Net with PPO Reinforcement Learning

In [10]:
####################################
# Policy net
####################################
# classes of reinforcement learning LLM networks (Step 4)
class policy_actor_LLM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.seq_pos_encoding = torch.arange(context_length, device=device) # positional encoding = {0,1,2,...,context_length-1}
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.transformer_blocks = nn.ModuleList([ TransformerBlock(d, context_length, num_heads) for _ in range(num_layers) ]) # multiple transformer block layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
        self.context_length = context_length
        self.padding = padding_int
        self.eos = eos_int
        # Note : No forward function is needed

# RL_LLM Policy network
policy_net = policy_actor_LLM(num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int).to(device)
num_param = number_param(policy_net)
print(f'Policy net (Step 4), initialized with SFT-LLM net (Step 2), num_param: {num_param/1e6}')
policy_net.load_state_dict(checkpoint_SFT_LLM['SFT_LLMnet_dict'])


####################################
# Value net
####################################
class value_critic_LLM(nn.Module):
    def __init__(self, SFT_LLM, d, context_length, padding_int, eos_int):
        super().__init__()
        self.SFT_LLM = SFT_LLM # SFT-LLM backbone 
        self.value = nn.Sequential(nn.Linear(d,4*d), nn.ReLU(), nn.Linear(4*d,1)) # value prediction layer
        self.cls_token_input = torch.tensor([0]).long().to(device)
        self.cls_token_embed = nn.Sequential( nn.Embedding(1, 4*d), nn.ReLU(), nn.Linear(4*d, d) )
        self.context_length = context_length
        self.padding = padding_int
        self.eos = eos_int
    def forward(self, state): 
        sequence = [ self.padding.item() ] * max(len(state),self.context_length)
        sequence[-len(state):] = state
        offset_start = max( self.context_length - len(state), 0 )
        sequence_torch = torch.tensor(sequence).to(device)
        context = sequence_torch[-self.context_length:] # crop if necessary the generated sequence with the context_length of the network
        context = context.unsqueeze(0).to(device) # [bs=1, len(state), d]
        H = self.SFT_LLM.token2vec(context) + self.SFT_LLM.PE_embedding(self.SFT_LLM.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # [bs=1, len(context), d] [1, 15, 384]
        h_cls = self.cls_token_embed(self.cls_token_input).expand(1,-1,-1) # [1, 1, d]  
        H = torch.cat([H, h_cls], dim=1) # [bs=1, len(context)+1, d]
        for transformer_block in self.SFT_LLM.transformer_blocks: H = transformer_block(H) # [bs=1, len(context)+1, d]
        token_scores = H[:,-1,:] # extract last token scores to predict value [bs=1, d]
        critic_value = self.value(token_scores).squeeze() # compute critic_value, size=[]
        return critic_value
        
# Initialize RL-LLM with pre-trained SFT-LLM network (Step 2)
value_net = value_critic_LLM(SFT_LLMnet, d, context_length, padding_int, eos_int).to(device)
num_param = number_param(value_net)
print(f'Value net (Step 4), initialized with SFT-LLM net (Step 2), num_param: {num_param/1e6}')


Policy net (Step 4), initialized with SFT-LLM net (Step 2), num_param: 10.761345
Value net (Step 4), initialized with SFT-LLM net (Step 2), num_param: 11.945986


## Reinforcement learning LM network (Step 4)


## Step 1 : Rollout one episode

In [11]:
# Rollout one episode (RL) = Generate a sequence of tokens from a prompt (LLM)

#  s_0 = [ prompt ] => a_0 (first generated token, t=0)   ~ policy_net(s_0)   => r_0 = 0(no reward), s_1 = [ prompt, a_0 ] 
#                   => a_1 (second generated token, t=1)  ~ policy_net(s_1)   => r_1 = 0(no reward), s_2 = [ prompt, a_0, a_1 ]
#                   ...
#                   => a_t (generated token t)            ~ policy_net(s_t)   => r_t = 0(no reward), s_t+1 = [ prompt, a_0, a_1, ... a_t ]
#                   ...
#                   => a_T-1 (last generated token t=T-1) ~ policy_net(s_T-1) => r_T-1 = Reward Model(s_T-1), and 
#                                                                                                    s_T = [ prompt, a_0, a_1, ... a_T-1=EOS ] 
#                   End of episode (RL) = End of generated sequence (LLM) with a_T-1 = EOS token

# a_t ~ net_policy(s_t) = p_t (probability over the dictionary of tokens), s_t+1 = s_t + a_t, r_t = 0 or RM(s_T-1) if a_t=EOS
def rollout_episode_LLM(policy_net, value_net, reward_model, SFT_LLMnet, prompt, max_len=15, temperature=1.0, kl_beta=0.01):
    policy_net.eval()
    value_net.eval()
    reward_model.eval()
    generated_seq = [ policy_net.padding.item() ] * max(prompt.size(0),policy_net.context_length) # initiliaze with padding, [127, ..., 127]
    generated_seq[-prompt.size(0):] = prompt.tolist() # [127, ..., 127, 102, 103, 112, 14]
    offset_start = max( policy_net.context_length - prompt.size(0), 0 ) # 40
    states = []; next_states = []; actions = []; rewards = []; log_probs = []; values = []; dones = []
    eos_token = policy_net.eos.item()
    for idx in range(max_len): # number of auto-regressive steps = number of generated tokens = number of steps in RL
        s_t = generated_seq[offset_start:].copy() # s_t = [102, 103, 112, 14]
        states.append(s_t) # [ s_0=[102, 103], s_1=[102, 103, 112], s_2=[102, 103, 112, 14], ...]

        def generate_next_token(policy_net, generated_seq, temperature):
            context = generated_seq[-policy_net.context_length:].unsqueeze(0) # size=[1, context_length]
            H = policy_net.token2vec(context) + policy_net.PE_embedding(policy_net.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # size=[batch_size, context_length, d]
            for transformer_block in policy_net.transformer_blocks: H = transformer_block(H) # size=(1, context_length, d)
            token_scores = H[:,-1,:] # extract last token to predict the next one, size=[1, d]
            token_scores = policy_net.token_prediction(token_scores) # compute scores, size=[1, num_tokens]
            token_probs = torch.softmax(token_scores/ temperature, dim=1) # compute probs, size=[num_tokens]
            token_probs = torch.nan_to_num(token_probs, nan=0.0)
            token_probs = token_probs + 1e-10 
            token_probs = token_probs / token_probs.sum(dim=1, keepdim=True)
            next_token = torch.multinomial(token_probs, num_samples=1) # sample next token, size=[1, 1]
            next_token = next_token.squeeze().item()
            token_probs = token_probs.squeeze() # size=[num_tokens]
            next_token_prob = token_probs[next_token] # probability of next token, size=[batch_size]
            next_token_prob = next_token_prob.item()
            return next_token, next_token_prob

        generated_seq_torch = torch.tensor(generated_seq).to(device) # [127, ..., 127, 102, 103, 112, 14]
        next_token, next_token_prob = generate_next_token(policy_net, generated_seq_torch, temperature)
        generated_seq.append(next_token) # s_t+1 = [127, ..., 127, 102, 103, 112, 14, 45]
        next_state = generated_seq[offset_start:].copy() # s_t+1 = [102, 103, 112, 14, 45]
        next_states.append(next_state.copy()) # [s_0, s_1, .., s_t, .., s_T], s_T = final generated sequence (last token is EOF)
        actions.append(next_token) # [a_0, a_1, .., a_t, .., a_T], a_T = EOF token
        log_prob = torch.log(torch.tensor(next_token_prob) + 1e-10).item() # log p_t
        log_probs.append(log_prob) # [log p_0, log p_1, ..., log p_T]

        # Get log prob from Reference Model (SFT) 
        with torch.no_grad():
            # Standard forward pass to get reference logits for the SAME context
            context = generated_seq_torch[-SFT_LLMnet.context_length:].unsqueeze(0)
            H = SFT_LLMnet.token2vec(context) + SFT_LLMnet.PE_embedding(SFT_LLMnet.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # size=[batch_size, context_length, d]
            for transformer_block in SFT_LLMnet.transformer_blocks: H = transformer_block(H) # size=(1, context_length, d)
            token_scores_ref = H[:,-1,:] # extract last token to predict the next one, size=[1, d]
            token_scores_ref = SFT_LLMnet.token_prediction(token_scores_ref) # compute scores, size=[1, num_tokens]
            token_probs_ref = torch.softmax(token_scores_ref/ temperature, dim=1) # compute probs, size=[num_tokens]
            token_probs_ref = torch.nan_to_num(token_probs_ref, nan=0.0)
            token_probs_ref = token_probs_ref + 1e-10 
            token_probs_ref = token_probs_ref / token_probs_ref.sum(dim=1, keepdim=True)
            next_token_prob_ref = token_probs_ref.squeeze()[next_token] # Probability of the action we JUST took, according to the SFT model
            log_prob_ref = torch.log(next_token_prob_ref + 1e-10).item()

        # Calculate Token-Level KL Penalty : kl = log(policy / reference)
        kl_div = log_prob - log_prob_ref
        kl_penalty = kl_beta * kl_div

        # Reward
        value = value_net(s_t) # critic value = Q(s_t) = Q_t (estimated total discounted reward)
        value = value.item()
        values.append(value) # [Q_0, Q_1, ..., Q_T]
        if next_token == eos_token: # end of episode
            with torch.no_grad():
                reward = reward_model(generated_seq[:-1]).detach().item() # use reward model to evaluate the generated_seq (without EOS token), r_T-1 = RM(s_T-1))
            rewards.append(reward - kl_penalty)
            dones.append(1.0) # done=1 for end of episode
            break
        else:
            rewards.append(0.0 - kl_penalty) # no reward for intermediate times, r_t = 0 for t < T
            dones.append(0.0) # done=0 if not end of episode

    return states, next_states, actions, rewards, log_probs, values, dones
    

# extract one train prompt
idx = torch.randint(0, len(list_prompt_RL), (1,)).item()
# idx = 10
print('idx:', idx)
prompt = list_prompt_RL[idx] 
max_len = 15
temperature = 4
temperature = 1.2
kl_beta = 0 * 0.05 
states, next_states, actions, rewards, log_probs, values, dones = rollout_episode_LLM(policy_net, value_net, reward_model, SFT_LLMnet, prompt, max_len, temperature, kl_beta)
print('last state', states[-1])
print('last next_state', next_states[-1])
print('actions', actions)
print('rewards', rewards)
print('log_probs',log_probs)
print('values', values)
print('dones',dones)


idx: 24
last state [113, 51, 114, 115, 116, 117, 107, 29, 115, 108, 116, 110, 12, 115, 111, 112, 118, 119, 115, 104, 105, 29, 37, 38, 67, 79, 39]
last next_state [113, 51, 114, 115, 116, 117, 107, 29, 115, 108, 116, 110, 12, 115, 111, 112, 118, 119, 115, 104, 105, 29, 37, 38, 67, 79, 39, 128]
actions [29, 37, 38, 67, 79, 39, 128]
rewards [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.517412185668945]
log_probs [-0.004832601174712181, -0.013974220491945744, -0.004567607771605253, -0.0002588964707683772, -0.005289512220770121, -0.00881414394825697, -0.008743911981582642]
values [0.33965837955474854, 0.2525242269039154, 0.9062910079956055, 0.13858696818351746, 0.34389179944992065, -0.2984369695186615, -1.2066681385040283]
dones [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]


## Step 2 : Generate a sequence using the policy net

In [12]:
# a_t ~ net_policy(s_t) = p_t (probability over the dictionary of tokens), s_t+1 = s_t + a_t, r_t = 0 or RM(s_T-1) if a_t=EOS
def generate_sequence_rl(policy_net, prompt, temperature, max_len):
    policy_net.eval()
    with torch.no_grad():
        generated_seq = [ policy_net.padding.item() ] * max(prompt.size(0),policy_net.context_length) # initiliaze with padding, [127, ..., 127]
        generated_seq[-prompt.size(0):] = prompt.tolist() # [127, ..., 127, 102, 103, 112, 14]
        offset_start = max( policy_net.context_length - prompt.size(0), 0 ) # 40
        states = []; next_states = []; actions = []; rewards = []; log_probs = []; values = []; dones = []
        eos_token = policy_net.eos.item()
        for idx in range(max_len): # number of auto-regressive steps = number of generated tokens = number of steps in RL
            s_t = generated_seq[offset_start:].copy() # s_t = [102, 103, 112, 14]
            states.append(s_t) # [ s_0=[102, 103], s_1=[102, 103, 112], s_2=[102, 103, 112, 14], ...]
            def generate_next_token(policy_net, generated_seq, temperature):
                context = generated_seq[-policy_net.context_length:].unsqueeze(0) # size=[1, context_length]
                H = policy_net.token2vec(context) + policy_net.PE_embedding(policy_net.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # size=[batch_size, context_length, d]
                for transformer_block in policy_net.transformer_blocks: H = transformer_block(H) # size=(1, context_length, d)
                token_scores = H[:,-1,:] # extract last token to predict the next one, size=[1, d]
                token_scores = policy_net.token_prediction(token_scores) # compute scores, size=[1, num_tokens]
                token_probs = torch.softmax(token_scores/ temperature, dim=1) # compute probs, size=[num_tokens]
                token_probs = torch.nan_to_num(token_probs, nan=0.0)
                token_probs = token_probs + 1e-10 
                token_probs = token_probs / token_probs.sum(dim=1, keepdim=True)
                next_token = torch.multinomial(token_probs, num_samples=1) # sample next token, size=[1, 1]
                next_token = next_token.squeeze().item()
                token_probs = token_probs.squeeze() # size=[num_tokens]
                next_token_prob = token_probs[next_token] # probability of next token, size=[batch_size]
                next_token_prob = next_token_prob.item()
                return next_token, next_token_prob
            generated_seq_torch = torch.tensor(generated_seq).to(device) # [127, ..., 127, 102, 103, 112, 14]
            next_token, _ = generate_next_token(policy_net, generated_seq_torch, temperature)
            generated_seq.append(next_token) # s_t+1 = [127, ..., 127, 102, 103, 112, 14, 45]
            next_state = generated_seq[offset_start:].copy() # s_t+1 = [102, 103, 112, 14, 45]
            next_states.append(next_state.copy()) # [s_0, s_1, .., s_t, .., s_T], s_T = final generated sequence (last token is EOF)
            actions.append(next_token) # [a_0, a_1, .., a_t, .., a_T], a_T = EOF token
            if next_token == eos_token: # end of episode
                with torch.no_grad():
                    reward = reward_model(generated_seq[:-1]).detach().item() # use reward model to evaluate the generated_seq (without EOS token), r_T-1 = RM(s_T-1))
                break
    prompt_response = s_t
    only_response = s_t[len(prompt):]
    reward, exact_response = reward_model_oracle(prompt, only_response)
    return prompt_response, only_response, reward, exact_response



## Step 3 : Train Policy and Value networks with PPO

In [13]:
# PPO parameters
opt_parameters = {}
opt_parameters['gamma'] = 0.99
opt_parameters['beta'] = 0.97
opt_parameters['clip_value'] = 0.2
opt_parameters['entropy_value'] = 0.01 
opt_parameters['num_iter_value_loss']  = 2
opt_parameters['num_iter_policy_loss'] = 2 
gamma = opt_parameters['gamma']
beta = opt_parameters['beta']
num_iter_value_loss = opt_parameters['num_iter_value_loss']
num_iter_policy_loss = opt_parameters['num_iter_policy_loss']
clip_value = opt_parameters['clip_value']
entropy_value = opt_parameters['entropy_value']

# Episode parameter
temperature = 1.2 
kl_beta = 0.01 

# Policy and Value networks
init_lr = 1e-5 
final_lr = 1e-7 
warmup_steps = 10
num_batch = 1 # TODO : MINI-BATCH  
num_epochs = len(list_prompt_RL) 
total_steps = num_batch * num_epochs  
policy_optimizer = torch.optim.AdamW(policy_net.parameters(), lr=init_lr) # Optimizer
value_optimizer = torch.optim.AdamW(value_net.parameters(), lr=init_lr) # Optimizer

# Learning rate strategy
def lr_lambda(current_step):
    # 1. Linear Warmup Phase
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    # 2. Cosine Annealing Phase
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    # Clamp progress at 1.0 to avoid errors if training goes longer than expected
    progress = min(1.0, progress)
    cosine_decay = 0.5 * (1.0 + torch.cos(torch.tensor(torch.pi * progress)).item()) # Standard cosine
    # Calculate the ratio between final_lr and init_lr to ensure we hit the floor
    lr_ratio = final_lr / init_lr
    # The multiplier starts at 1.0 and ends at lr_ratio
    return lr_ratio + (1.0 - lr_ratio) * cosine_decay
policy_scheduler = torch.optim.lr_scheduler.LambdaLR(policy_optimizer, lr_lambda=lr_lambda) # Scheduler
value_scheduler = torch.optim.lr_scheduler.LambdaLR(value_optimizer, lr_lambda=lr_lambda) # Scheduler

# Print
print(f'temperature : {temperature}, kl_beta : {kl_beta}, entropy_value : {opt_parameters["entropy_value"]}')
print(f'num_iter_value_loss : {opt_parameters["num_iter_value_loss"]}, num_iter_policy_loss : {opt_parameters["num_iter_policy_loss"]}')
print(f'init_lr : {init_lr}, final_lr : {final_lr}')

# For saving policy network
print('checkpoint :',"step4_checkpoint_RL_LLM_" + time_stamp + '.pkl', '\n')
checkpoint_dir = os.path.join("checkpoint")
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

# Train network to predict a response from a prompt
start = time.time()
num_prompts = len(list_prompt_RL)
running_policy_loss = 0.0 
running_value_loss = 0.0 
shuffle = torch.randperm(num_prompts)
for idx in range(num_prompts):

    
    ########################################
    # Rollout one episode 
    ########################################
    idx_shuffle = shuffle[idx].item()
    prompt = list_prompt_RL[idx_shuffle] 
    states, next_states, actions, rewards, log_probs, values, dones = \
    rollout_episode_LLM(policy_net, value_net, reward_model, SFT_LLMnet, prompt, 15, temperature, kl_beta)

    
    ########################################
    # Update value network
    ########################################
    #  Step 1 : Compute discounted rewards at each time t for the whole episode
    discounted_rewards = [] # collect discounted rewards
    discounted_reward_t_plus_one = 0.0 # r_t+1=r_T=0 (end-of-episode, no more reward) for t=T-1, T=length of episode
    for r_t in rewards[::-1]: # compute discounted reward (dr) in reverse time, from t=T-1, T-2, ..., 0
                              # i.e. rewards[::-1] = [r_T-1, r_T-2, r_T-3, ..., r_1, r_0] (reverse list)
                              # why reversed? to use fast recursive equation : dr_t = r_t + g * dr_t+1, g = gamma
        discounted_reward_t = r_t + gamma * discounted_reward_t_plus_one
         #             dr_t = r_t + g * dr_{t+1}
         #  t=T-1 => dr_T-1 = r_T-1 + g * dr_T=0 = r_T-1
         #  t=T-2 => dr_T-2 = r_T-2 + g * dr_T-1
         #                  = r_T-2 + g * r_T-1
         #  t=T-3 => dr_T-3 = r_T-3 + g * dr_T-2
         #                  = r_T-3 + g * (r_T-2 + g* r_T-1)
         #                  = r_T-3 + g * r_T-2 + g^2* r_T-1
         #                  etc
         #  t=0   =>   dr_0 = r_0 + g * r_1 + ... + g^T-1 r_T-1
        discounted_rewards.append(discounted_reward_t) # size=[t+1], discounted_rewards = [dr_T-1, dr_T-2, ..., dr_0] at the last time t=0
        discounted_reward_t_plus_one = discounted_reward_t # time-update discounted reward
    discounted_rewards.reverse() # Reverse the list back to original time order, i.e. t=0 to T-1
                                 # discounted_rewards = [dr_0, dr_1, ..., dr_T-1]
    discounted_rewards = torch.tensor(discounted_rewards, dtype=torch.float32).to(device)

    #  Step 2 : Compute loss between predicted value and the episode discounted reward
    #           Note : states and discounted_rewards collected from the episode are fixed during optimization
    #
    # Loss_value = mean_{t=0}^{T-1} || V_phi_t - dr_t ||^2, V_phi_t = net_value(state_t), scalar
    #
    #                                       with dr_t = sum_{l=0} gamma^l r_t+l
    #
    # min Loss_value => V_phi_t ~ dr_t
    #
    for _ in range(num_iter_value_loss):
        value_optimizer.zero_grad()
        values_pred = []
        for state in states:
            values_pred.append(value_net(state).squeeze())
        values_pred = torch.stack(values_pred) # size=[T] value = value_net(s_t), t=0,1,..T-1
        value_loss = ( (values_pred - discounted_rewards.detach())**2 ).mean() # MSE, scalar
        value_loss.backward()
        torch.nn.utils.clip_grad_norm_(value_net.parameters(), 1.0)
        value_optimizer.step()
    value_scheduler.step()
    value_loss = value_loss.detach().item() 

    
    ########################################
    # Update state-policy network
    ########################################
    #  Step 1 : Compute advantage values at each time step for the whole episode
    Q_t = values # Q_t = [Q_0, Q_1, Q_2, ..., Q_T-1], size=[T]
    Q_t_plus_one = Q_t[1:] + [0.0] # Q_t+1 = [Q_1, Q_2, ..., Q_T=0], size=[T]
    Q_t = torch.tensor(Q_t).to(device)
    Q_t_plus_one = torch.tensor(Q_t_plus_one).to(device)
    rewards = torch.tensor(rewards).to(device)
    dones = torch.tensor(dones).to(device)
    # deltas = rewards + gamma * Q_t_plus_one - Q_t 
    deltas = rewards + gamma * Q_t_plus_one * (1 - dones) - Q_t # deltas = [delta_0, delta_1, ..., delta_T-1], 
                                                                # delta_t = r_t + gamma * Q_t+1 - Q_t, size=[T]
                                                                # (1 - dones) is used to handle terminal states explicitly
    advantages = torch.zeros_like(deltas).to(device)
    advantage_t_plus_one = 0.0 # A_t+1=0 for t=T-1
    for t in range(len(deltas))[::-1]: # compute the generalized advantage estimator at each time step
                                       # deltas[::-1] = [delta_T-1, delta_T-2, ..., delta_1, delta_0]
                                       # why reversed? bc recursive equation is fast to compute : A_t = delta_t + gamma * beta * A_t+1
        advantage_t = deltas[t] + gamma * beta * advantage_t_plus_one # calculate GAE recursively
        advantages[t] = advantage_t  # size=[t+1], advantages = [A_0, A_1, ..., A_T-1]
        advantage_t_plus_one = advantage_t
    # Normalize the advantages, better gradient scale
    advantages = ( advantages - advantages.mean() ) / ( advantages.std() + 1e-8 )
    
    #  Step 2 : Compute clipped trust region loss for policy network
    #           Note : states, actions, previous log_probs and advantages collected from the episode are fixed during optimization
    #
    # Loss_policy = - mean_{t=0}^{T-1} ( min( ratio_t * A_t , clip(ratio_t) * A_t ) )
    #
    #             with ratio_t = Policy_Net(a_t|s_t) / Policy_Net_previous(a_t|s_t), Policy_Net = Probability
    #
    #                      A_t = sum_{l=0} (gamma * beta)^l delta_t+l, delta_t = r_t + gamma * Q_t+1 - Q_t
    #
    #            clip(ratio_t) = max( 1 - clip_value , min(ratio_t, 1 + clip_value) )
    #
    # min Loss_policy => max Policy_Net(a_t|s_t) * A_t with Policy_Net better than Policy_Net_previous
    #
    #                                                   and clipping limits the variations of Policy_Net
    #
    #                    max Policy_Net(a_t|s_t) * A_t => select action that long-term reward
    
    log_probs_previous = torch.tensor(log_probs).to(device) # use log_probs from episode (fixed during optimization), size=[T]
    actions = torch.tensor(actions).to(device)
    action_probs = []
    for _ in range(num_iter_policy_loss):
        policy_optimizer.zero_grad()
        action_probs = []
        for state in states:
            context = state[-policy_net.context_length:] # crop if necessary the generated sequence with the context_length of the network
            context = torch.tensor(context, dtype=torch.long).unsqueeze(0).to(device) # [bs=1, len(state), d]
            H = policy_net.token2vec(context) + policy_net.PE_embedding(policy_net.seq_pos_encoding[:context.size(1)]).unsqueeze(0) # [bs=1, len(state), d]
            for transformer_block in policy_net.transformer_blocks: H = transformer_block(H) # [bs=1, len(state), d]
            token_scores = H[:,-1,:] # extract last token to predict the next one, size=[bs=1, d]
            token_scores = policy_net.token_prediction(token_scores) # compute scores, size=[bs=1, num_tokens=129]
            action_prob = torch.softmax(token_scores/temperature, dim=1) # compute probs, size=[bs=1, num_tokens]
            action_prob = torch.nan_to_num(action_prob, nan=0.0)
            action_prob = action_prob + 1e-10 
            action_prob = action_prob / action_prob.sum(dim=1, keepdim=True)
            action_probs.append(action_prob.squeeze())
        action_probs = torch.stack(action_probs)
        log_probs = torch.log( action_probs[torch.arange(actions.size(0)), actions] + 1e-10 ) # use actions from episode (no sampling), size=[T]
        policy_ratio = torch.exp( log_probs - log_probs_previous.detach()) # ratio between new optimized policy and previous one, size=[T]
        clipped_ratio = policy_ratio.clamp(1.0 - clip_value, 1.0 + clip_value) # clipped ratio to allow small changes only, size=[T]
        loss_proximal = - torch.min( policy_ratio * advantages , clipped_ratio * advantages ).mean() # select the loss with smallest change, scalar
        loss_entropy = - (action_probs * torch.log(action_probs + 1e-10)).sum(dim=-1).mean()
        # entropy regularization of the policy model, it prevents the model to "mode collapse" : at the beginning of the training, 
        # the model can become too "confident" too quickly, then the gradient will vanish and the model will stop learning
        policy_loss = loss_proximal - entropy_value * loss_entropy 
        policy_loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
        policy_optimizer.step()
    policy_scheduler.step()
    policy_loss = policy_loss.detach().item() # for print
    
    # Track losses
    running_policy_loss += policy_loss
    running_value_loss += value_loss

    # Print
    if not idx%1 and idx>0:
        mean_policy_loss = running_policy_loss/(idx+1)
        mean_value_loss = running_value_loss/(idx+1)
        print(f"idx : {idx}, time(min) : {(time.time()-start)/60:.3f}, policy_lr : {policy_optimizer.param_groups[0]['lr']:.7f}, value_lr : {value_optimizer.param_groups[0]['lr']:.7f}, mean_policy_loss : {mean_policy_loss:.4f}, mean_value_loss : {mean_value_loss:.4f}")

    # print one generated sequence and save policy network
    if not idx%10 and idx>0:
        prompt_response, only_response, reward, exact_response = generate_sequence_rl(policy_net, prompt, temperature, 15)
        print('generated_response:', func_tokens2str(func_indices2tokens(prompt_response)))
        print('generated_response_only :', [x for x in func_tokens2str(func_indices2tokens(only_response)).split()])
        print('exact_response:', exact_response)
        print('reward_model :', reward)
        # save checkpoint
        torch.save({
            'idx': idx,
            'tot_time': time.time()-start,
            'opt_parameters': opt_parameters,
            'mean_policy_loss': mean_policy_loss,
            'mean_value_loss': mean_value_loss,
            'net_parameters': net_parameters,
            'policy_net_dict': policy_net.state_dict(),
            }, '{}.pkl'.format(checkpoint_dir + "/step4_checkpoint_RL_LLM_" + time_stamp ))



temperature : 1.2, kl_beta : 0.01, entropy_value : 0.01
num_iter_value_loss : 2, num_iter_policy_loss : 2
init_lr : 1e-05, final_lr : 1e-07
checkpoint : step4_checkpoint_RL_LLM_26-02-07--18-48-32.pkl 

idx : 1, time(min) : 0.066, policy_lr : 0.0000020, value_lr : 0.0000020, mean_policy_loss : 0.1743, mean_value_loss : 32.2104
idx : 2, time(min) : 0.077, policy_lr : 0.0000030, value_lr : 0.0000030, mean_policy_loss : 0.0963, mean_value_loss : 35.8626
idx : 3, time(min) : 0.091, policy_lr : 0.0000040, value_lr : 0.0000040, mean_policy_loss : 0.1203, mean_value_loss : 35.4138
idx : 4, time(min) : 0.104, policy_lr : 0.0000050, value_lr : 0.0000050, mean_policy_loss : 0.1138, mean_value_loss : 34.5953
idx : 5, time(min) : 0.125, policy_lr : 0.0000060, value_lr : 0.0000060, mean_policy_loss : 0.1321, mean_value_loss : 32.9343
idx : 6, time(min) : 0.148, policy_lr : 0.0000070, value_lr : 0.0000070, mean_policy_loss : 0.1468, mean_value_loss : 31.3065
idx : 7, time(min) : 0.158, policy_lr : 0.

## Load pre-trained RL-LLM network

In [15]:
# Initialize RL-LLM with pre-trained SFT-LLM network (Step 2)
# checkpoint_file = 'checkpoint/step4_checkpoint_RL_LLM_' + str(time_stamp) + '.pkl'
# checkpoint_file = 'checkpoint/step4_checkpoint_RL_LLM_26-02-07--18-48-32.pkl'
import os
checkpoint_file = 'checkpoint/step4_checkpoint_RL_LLM_26-02-07--18-48-32_100_10M.pkl' # GPU pre-trained
print('checkpoint_file:', checkpoint_file, '\n')
if not os.path.exists(checkpoint_file):
    print(f'Downloading {checkpoint_file} ...')
    dropbox_url = "https://www.dropbox.com/scl/fi/e0wy85i398ipp81i6bcl9/step4_checkpoint_RL_LLM_26-02-07-18-48-32_100_10M.pkl?rlkey=6ys97lo7l87ex6yxixsbdphq2&dl=1"
    !wget -q "{dropbox_url}" -O "{checkpoint_file}"
print('checkpoint_file:', checkpoint_file)
checkpoint = torch.load(checkpoint_file, map_location=device)
net_parameters = checkpoint['net_parameters']
num_tokens = net_parameters['num_tokens']
d = net_parameters['d']
num_heads = net_parameters['num_heads']
context_length = net_parameters['context_length']
num_layers = net_parameters['num_layers']
padding_int = net_parameters['padding_int']
eos_int = net_parameters['eos_int']
padding_int = torch.tensor([func_tokens2indices('<PAD>'.split())[0]]).to(device) # end-of-sentence token for batch
eos_int = torch.tensor([func_tokens2indices('<EOS>'.split())[0]]).to(device) # end-of-sentence token for batch
class policy_actor_LLM(nn.Module):
    def __init__(self, num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int):
        super().__init__()
        self.token2vec = token2vec(num_tokens, d) # token embedding layer
        self.seq_pos_encoding = torch.arange(context_length, device=device) # positional encoding = {0,1,2,...,context_length-1}
        self.PE_embedding = nn.Embedding(context_length, d) # positional encoding embedding layer
        self.transformer_blocks = nn.ModuleList([ TransformerBlock(d, context_length, num_heads) for _ in range(num_layers) ]) # multiple transformer block layers
        self.token_prediction = nn.Linear(d, num_tokens) # next token prediction layer
        self.context_length = context_length
        self.padding = padding_int
        self.eos = eos_int
        # Note : No forward function is needed
# RL_LLM Policy network
policy_net = policy_actor_LLM(num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int).to(device)
num_param = number_param(policy_net)
print(f'Policy net (Step 4)')
print('num_tokens: %d, d: %d, context_length: %d, num_heads: %d, num_layers: %d, padding_int: %d, eos_int: %d, num_param: %f' \
      % (num_tokens, d, context_length, num_heads, num_layers, padding_int, eos_int, num_param/1e6) ) 
policy_net.load_state_dict(checkpoint['policy_net_dict'])


# parameters for arithmetic series
s = torch.randint(low=0, high=max_value, size=(1,)).item() # starting integer of the series
d = torch.randint(low=1, high=10, size=(1,)).item() # value of common difference
n = torch.randint(low=5, high=15, size=(1,)).item() # number of element in the series
# generate prompt : sample a prompt between 3 candidate prompts
prompt = {}
prompt[1] = 'generate an arithmetic series with ' + str(n) + ' terms starting with value ' + str(s) + ' and common difference ' + str(d)
prompt[2] = 'make a series of arithmetic type which starts at ' + str(s) + ' with ' + str(n) + ' elements and ' + str(d) + ' common difference value'
prompt[3] = 'Let ' + str(n) + ' be the number of terms ' + str(s) + ' the starting number and ' + str(d) + ' the common difference then write the arithmetic series'
random_int = torch.randint(low=1, high=3+1, size=(1,)).item() # random number in {1,2,3}
prompt = prompt[random_int]
# convert from token to pytorch
prompt = [str(i) for i in prompt.split()] # convert a string into seq of tokens (w/ string type)
prompt = func_tokens2indices(prompt) # convert from token (str) to index (int)
prompt = torch.tensor(prompt) # convert to pytorch
print('prompt:', prompt)

# Generate a new sequence
max_len = 15
temperature = 1
prompt_response, only_response, reward, exact_response = generate_sequence_rl(policy_net, prompt, temperature, max_len)
print('generated_response:', func_tokens2str(func_indices2tokens(prompt_response)))
print('generated_response_only :', [x for x in func_tokens2str(func_indices2tokens(only_response)).split()])
print('exact_response:', exact_response)
print('reward_model :', reward)
        


checkpoint_file: checkpoint/step4_checkpoint_RL_LLM_26-02-07--18-48-32_100_10M.pkl 

checkpoint_file: checkpoint/step4_checkpoint_RL_LLM_26-02-07--18-48-32_100_10M.pkl
Policy net (Step 4)
num_tokens: 129, d: 384, context_length: 40, num_heads: 6, num_layers: 6, padding_int: 127, eos_int: 128, num_param: 10.761345
prompt: tensor([102, 103, 104, 105, 106,  90, 107, 108, 106, 109,  98, 110, 111, 112,
         11])
generated_response: generate an arithmetic series with 14 terms starting with value 35 and common difference 1 35 36 37 38 39 40 41 42 43 44 45 46 47 48
generated_response_only : ['35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48']
exact_response: [35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48]
reward_model : 7.0
